# Experiment 2 — GE-MolSG vs ElectroShape vs ESP-Sim (DUDE-Z retrieval)

Per-target retrieval performance (EF1%, BEDROC) for three methods:

- **GE-MolSG** — this package's WKS → hard k-NN BoF (`knn_histogram`) over a geo
  codebook, chi-squared-kernel retrieval.
- **ElectroShape** — 4D ElectroShape (oddt) on PDB conformers with **MMFF94**
  partial charges (OpenBabel); USR-style similarity.
- **ESP-Sim** — Crippen-O3A alignment + shape + **ML-charge ESP** (`espsim`),
  pairwise `shape + esp_ml` score.

Same workflow as Experiment 1: results are generated **iteratively per target**
and cached to disk, so a rerun skips completed targets. After all targets
finish, per-target means are summarised, two scatter/line plots (BEDROC and
EF1%) are produced, and a Wilcoxon signed-rank test (focused on GE-MolSG) is run.

### Inputs
```
<DATA_ROOT>/DUDE-Z/<TARGET>.tar.gz -> <TARGET>/ESP_Npy/{ligand,decoy}_<ID>.npy
                                       <TARGET>/PDB_Files/<ID>.pdb        (ElectroShape/ESP-Sim)
experiments/codebooks/ge_molsg_cb.npy   (GE-MolSG geo codebook)
<QUERIES_DIR>/<target>.csv              (column 'query' or 'queries')
```

### Environments
ElectroShape needs `oddt` + OpenBabel (`pybel`); ESP-Sim needs `espsim`. If a
method's libraries are unavailable, that method's per-target encode fails
gracefully (logged) and it is simply omitted from the comparison.


## 1 · Configuration

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import logging
import sys
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd

# ── Experiment paths (EDIT THESE) ────────────────────────────────────────────
TARGETS = [
    "AA2AR", "ABL1", "ACES", "ADA", "ADRB2", "AMPC", "ANDR", "CSF1R",
    "CXCR4", "DEF", "DRD4", "EGFR", "FA10", "FA7", "FABP4", "FGFR1",
    "FKB1A", "GLCM", "HDAC8", "HIVPR", "HMDH", "HS90A", "ITAL", "KIT",
    "KITH", "LCK", "MAPK2", "MK01", "MT1", "NRAM", "PARP1", "PLK1",
    "PPARA", "PTN1", "PUR2", "RENI", "ROCK1", "SRC", "THRB", "TRY1",
    "TRYB1", "UROK", "XIAP",
]  # comment out any targets you don't want to run
# Data: per-target archives <TARGET>.tar.gz are hosted in one Zenodo record.
# Set ZENODO_BASE_URL and each listed target is downloaded + extracted on demand
# (only the targets in TARGETS are fetched). Each <TARGET>.tar.gz extracts to
# <TARGET>/{ESP_Npy, ESP_Npy_MMFF94, PDB_Files}/.
ZENODO_BASE_URL = "https://zenodo.org/records/20547837/files"   # e.g. "https://zenodo.org/records/XXXXXXX/files"
DATA_ROOT    = Path("dude_z_data")   # local cache for downloaded/extracted targets
QUERIES_DIR  = Path("queries")
OUT_DIR      = Path("experiments_out/exp2_gemolsg_eshape_espsim")
CODEBOOK_DIR = Path("codebooks")
GE_CODEBOOK  = CODEBOOK_DIR / "ge_molsg_cb.npy"

METHODS = ["GE-MolSG", "ElectroShape", "ESP-Sim"]
METRICS = ["EF1%", "BEDROC"]

# ── GE-MolSG descriptor params ───────────────────────────────────────────────
K, EVALS, VAR, KNN, EW, LAP_NORM, BOF_KNN = 100, 100, 15, 100, 0.3, "normalized", 3

# ── ElectroShape config ──────────────────────────────────────────────────────
ESHAPE_CHARGE_MODEL = "mmff94"   # MMFF94 partial charges (OpenBabel)

FORCE = False

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT = OUT_DIR / "results"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(OUT_DIR / "exp2.log")],
    force=True,
)
log = logging.getLogger("exp2")
log.info("Experiment 2 — GE-MolSG vs ElectroShape(%s) vs ESP-Sim", ESHAPE_CHARGE_MODEL)

## 2 · Data / query helpers

In [ ]:
import subprocess
import urllib.request


def fetch_target(target, data_root):
    """Ensure <data_root>/<TARGET>/ exists, downloading from Zenodo if needed.

    Looks for an already-extracted ``<TARGET>/ESP_Npy`` first. If absent and
    ``ZENODO_BASE_URL`` is set, downloads ``<ZENODO_BASE_URL>/<TARGET>.tar.gz``
    and extracts it under ``data_root``. Each archive extracts to
    ``<TARGET>/{ESP_Npy, ESP_Npy_MMFF94, PDB_Files}/``.

    Returns the ``<TARGET>/`` directory.
    """
    data_root = Path(data_root)
    for cand in (data_root / target, data_root / target.upper(),
                 data_root / "DUDE-Z" / target):
        if (cand / "ESP_Npy").is_dir():
            return cand

    local_tar = None
    for tar in (data_root / f"{target}.tar.gz",
                data_root / "DUDE-Z" / f"{target}.tar.gz"):
        if tar.exists():
            local_tar = tar
            break

    if local_tar is None:
        if not ZENODO_BASE_URL:
            raise FileNotFoundError(
                f"No local data for '{target}' and ZENODO_BASE_URL is not set.")
        data_root.mkdir(parents=True, exist_ok=True)
        local_tar = data_root / f"{target}.tar.gz"
        url = f"{ZENODO_BASE_URL.rstrip('/')}/{target}.tar.gz"
        log.info("Downloading %s", url)
        try:
            subprocess.run(["curl", "-fSL", "-o", str(local_tar), url], check=True)
        except (FileNotFoundError, subprocess.CalledProcessError):
            urllib.request.urlretrieve(url, local_tar)

    log.info("Extracting %s", local_tar)
    with tarfile.open(local_tar, "r:gz") as tf:
        tf.extractall(data_root)

    for cand in (data_root / target, data_root / target.upper()):
        if (cand / "ESP_Npy").is_dir():
            return cand
    hits = list(data_root.rglob(f"{target}/ESP_Npy")) or list(data_root.rglob("ESP_Npy"))
    if hits:
        return hits[0].parent
    raise FileNotFoundError(f"Extracted '{target}' but found no ESP_Npy under {data_root}")


def resolve_target_dir(target, data_root, work=None):
    """Return (esp_npy_dir, target_root). target_root also holds
    ESP_Npy_MMFF94 and PDB_Files."""
    target_root = fetch_target(target, data_root)
    esp_dir = target_root / "ESP_Npy"
    if not esp_dir.is_dir():
        raise FileNotFoundError(f"No ESP_Npy for '{target}' under {target_root}")
    return esp_dir, target_root

def resolve_queries_csv(queries_dir, target):
    for name in (f"{target}.csv", f"{target.lower()}.csv", f"{target.upper()}.csv"):
        if (queries_dir / name).exists():
            return queries_dir / name
    raise FileNotFoundError(f"No queries CSV for '{target}' under {queries_dir}")

def load_query_ids(csv_path):
    """Read query IDs using whichever query column is present ('query'/'queries')."""
    df = pd.read_csv(csv_path)
    col = next((c for c in ("queries", "query") if c in df.columns), None)
    if col is None:
        col = df.columns[0] if df.shape[1] == 1 else None
    if col is None:
        raise ValueError(f"{csv_path}: no 'query'/'queries' column found")
    return [str(q).strip() for q in df[col].dropna()]

def list_surface_fns(surf_dir):
    return sorted([f for f in os.listdir(surf_dir) if f.endswith(".npy")], reverse=True)

## 3 · Retrieval metrics
EF1% and BEDROC (α=20). GE-MolSG uses a precomputed chi² matrix; ElectroShape/ESP-Sim use a per-query `sim_fn` (pairwise).

In [ ]:
from rdkit.ML.Scoring import Scoring

def _metrics_from_sim(sim, labels_rest):
    order = np.argsort(sim)[::-1]
    scores = np.column_stack([np.asarray(sim)[order], np.asarray(labels_rest)[order]])
    ef = Scoring.CalcEnrichment(scores, 1, [0.01])
    bedroc = Scoring.CalcBEDROC(scores, 1, 20)
    return {"EF1%": float(ef[0]), "BEDROC": float(bedroc)}

def retrieve_simfn(descs, fns, query_ids, sim_fn):
    """Generic retrieval: sim_fn(descs, i) -> similarities of i vs all others (i removed)."""
    labels = np.asarray([1 if f[0] == "l" else 0 for f in fns])
    stems = [f[:-6] if f.endswith(".npy") else f.rsplit(".", 1)[0] for f in fns]
    rows = []
    for qid in query_ids:
        hits = [j for j, s in enumerate(stems) if s == qid or s.endswith(qid)]
        if not hits:
            log.warning("  query '%s' not found; skipping", qid)
            continue
        i = hits[0]
        rest = [labels[z] for z in range(len(fns)) if z != i]
        sim = sim_fn(descs, i)
        m = _metrics_from_sim(sim, rest)
        m["RefMol"] = stems[i]
        rows.append(m)
    return pd.DataFrame(rows, columns=["RefMol", "EF1%", "BEDROC"])

## 4 · Method encoders
Each returns `(descs, sim_fn, fns)` so retrieval is uniform across methods.

In [ ]:
import ge_molsg as gm
from sklearn.metrics.pairwise import chi2_kernel

# bundled molsg / experiments dir on path (for any shared imports)
sys.path.insert(0, "scripts/experiments")

# ── GE-MolSG ─────────────────────────────────────────────────────────────────
def encode_gemolsg(esp_dir, target_root, fns):
    surfaces = [gm.load_surface_npy(str(esp_dir / f), name=f[:-4]) for f in fns]
    codebook = np.load(GE_CODEBOOK, allow_pickle=True)
    vecs = []
    for surf in surfaces:
        pts = surf.augmented_points(elec_weight=EW)
        W = gm.compute_affinity(pts, n_neighbors=KNN, backend="ckdtree",
                                adaptive_bw=True, square_distances=False)
        L = gm.graph_laplacian(W, laplacian_type=LAP_NORM)
        ev, evec = gm.compute_eigenpairs(L, n_components=K, drop_first=True,
                                         normalize_vectors=False, eigensolver="arpack")
        wks = np.nan_to_num(gm.wks([ev, evec], evals=EVALS, variance=VAR, l2=True))
        vecs.append(gm.knn_histogram(wks, codebook, knn=BOF_KNN))
    V = np.asarray(vecs)
    S = chi2_kernel(V)

    def sim_fn(_descs, i):
        keep = np.arange(S.shape[0]) != i
        return S[i][keep]
    return V, sim_fn, fns

# ── ElectroShape (MMFF94) ────────────────────────────────────────────────────
def encode_electroshape(esp_dir, target_root, fns):
    from openbabel import openbabel as ob
    from openbabel import pybel
    from oddt.shape import electroshape, usr_similarity

    pdb_dir = target_root / "PDB_Files"
    stems = [f[:-4] for f in fns]

    def _read_pdb_ob(fp):
        mol = ob.OBMol(); conv = ob.OBConversion(); conv.SetInFormat("pdb")
        conv.ReadFile(mol, str(fp))
        cm = ob.OBChargeModel.FindType(ESHAPE_CHARGE_MODEL)
        cm.ComputeCharges(mol)
        return pybel.Molecule(mol)

    descs, keep_fns = [], []
    for stem, f in zip(stems, fns):
        pdb = pdb_dir / f"{stem}.pdb"
        if not pdb.exists():
            log.warning("  [ElectroShape] missing PDB %s", pdb); descs.append(None); keep_fns.append(f); continue
        try:
            descs.append(electroshape(_read_pdb_ob(pdb)))
        except Exception as e:
            log.warning("  [ElectroShape] %s failed: %r", stem, e); descs.append(None)
        keep_fns.append(f)

    def sim_fn(d, i):
        ref = d[i]
        return np.asarray([
            usr_similarity(ref, d[z]) if (d[z] is not None and ref is not None) else 0.0
            for z in range(len(d)) if z != i
        ])
    return descs, sim_fn, keep_fns

# ── ESP-Sim (ML-charge) ──────────────────────────────────────────────────────
def encode_espsim(esp_dir, target_root, fns):
    from copy import deepcopy
    from rdkit import Chem
    from rdkit.Chem import rdMolAlign, rdMolDescriptors
    from espsim import GetShapeSim, GetEspSim
    from espsim.helpers import mlCharges

    pdb_dir = target_root / "PDB_Files"
    stems = [f[:-4] for f in fns]

    descs, keep_fns = [], []
    for stem, f in zip(stems, fns):
        pdb = pdb_dir / f"{stem}.pdb"
        m = Chem.rdmolfiles.MolFromPDBFile(str(pdb), removeHs=False, sanitize=True) if pdb.exists() else None
        if m is None:
            log.warning("  [ESP-Sim] unparseable/missing %s", stem); descs.append(None); keep_fns.append(f); continue
        try:
            charges = mlCharges([m])
            descs.append([m, charges])
        except Exception as e:
            log.warning("  [ESP-Sim] charge calc failed %s: %r", stem, e); descs.append(None)
        keep_fns.append(f)

    def _align_crippen(prb, ref):
        pC = rdMolDescriptors._CalcCrippenContribs(prb)
        rC = rdMolDescriptors._CalcCrippenContribs(ref)
        rdMolAlign.GetCrippenO3A(prb, ref, pC, rC, 0, 0).Align()

    def _score(prb_e, ref_e):
        if prb_e is None or ref_e is None:
            return 0.0
        prb = deepcopy(prb_e[0]); ref = deepcopy(ref_e[0])
        try:
            _align_crippen(prb, ref)
            shape = GetShapeSim(prb, ref)
            esp = GetEspSim(prb, ref, 0, 0, prbCharge=prb_e[1], refCharge=ref_e[1],
                            renormalize=True, metric="tanimoto")
            return shape + esp
        except Exception:
            return 0.0

    def sim_fn(d, i):
        ref = d[i]
        return np.asarray([_score(d[z], ref) for z in range(len(d)) if z != i])
    return descs, sim_fn, keep_fns

ENCODERS = {
    "GE-MolSG": encode_gemolsg,
    "ElectroShape": encode_electroshape,
    "ESP-Sim": encode_espsim,
}

## 5 · Per-target driver (iterative, cached)

In [ ]:
def run_target(target):
    out_by_method = {}
    esp_dir = target_root = base_fns = query_ids = None
    for method in METHODS:
        csv_path = RESULTS_ROOT / method / f"{target}.csv"
        if csv_path.exists() and not FORCE:
            log.info("[%s | %s] cached -> %s", target, method, csv_path)
            out_by_method[method] = pd.read_csv(csv_path); continue
        if esp_dir is None:
            esp_dir, target_root = resolve_target_dir(target, DATA_ROOT, OUT_DIR / "_work")
            base_fns = list_surface_fns(esp_dir)
            query_ids = load_query_ids(resolve_queries_csv(QUERIES_DIR, target))
            n_lig = sum(f[0] == "l" for f in base_fns)
            log.info("[%s] %d surfaces (%d ligands, %d decoys), %d queries",
                     target, len(base_fns), n_lig, len(base_fns) - n_lig, len(query_ids))
        log.info("[%s | %s] encoding ...", target, method)
        try:
            descs, sim_fn, fns = ENCODERS[method](esp_dir, target_root, base_fns)
        except Exception as e:
            log.error("[%s | %s] encoding failed: %r", target, method, e)
            out_by_method[method] = pd.DataFrame(columns=["RefMol", "EF1%", "BEDROC"]); continue
        log.info("[%s | %s] retrieval ...", target, method)
        df = retrieve_simfn(descs, fns, query_ids, sim_fn)
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(csv_path, index=False)
        log.info("[%s | %s] %d queries  mean EF1%%=%.3f  mean BEDROC=%.3f -> %s",
                 target, method, len(df),
                 df["EF1%"].mean() if len(df) else float("nan"),
                 df["BEDROC"].mean() if len(df) else float("nan"), csv_path)
        out_by_method[method] = df
    return out_by_method

sampled_data = {m: {} for m in METHODS}
for ti, target in enumerate(TARGETS, 1):
    log.info("==== target %d/%d : %s ====", ti, len(TARGETS), target)
    try:
        by_method = run_target(target)
    except FileNotFoundError as e:
        log.error("skipping %s: %s", target, e); continue
    for method, df in by_method.items():
        if len(df):
            sampled_data[method][target] = df
log.info("encoding + retrieval complete")

## 6 · Summary

In [ ]:
rows = []
for method in sampled_data:
    for t in TARGETS:
        if t in sampled_data[method] and len(sampled_data[method][t]):
            df = sampled_data[method][t]
            rows.append(dict(method=method, target=t,
                             mean_EF1=df["EF1%"].mean(),
                             mean_BEDROC=df["BEDROC"].mean(), n_queries=len(df)))
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary_per_target.csv", index=False)
log.info("wrote summary_per_target.csv (%d rows)", len(summary))
summary

## 7 · Scatter / line plots — BEDROC and EF1%

In [ ]:
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", font_scale=1.0)
except Exception:
    pass

COLOUR_MAP = {"GE-MolSG": "#0072B2", "ElectroShape": "#FF6B6B", "ESP-Sim": "#E69F00"}
METHOD_MARKERS = {"GE-MolSG": "s", "ElectroShape": "v", "ESP-Sim": "^"}
SCATTER_METHODS = ["GE-MolSG", "ElectroShape", "ESP-Sim"]

def scatter(metric):
    tsorted = sorted(TARGETS)
    x_pos = np.arange(len(tsorted))
    y_max = 50.0 if metric == "EF1%" else 0.9
    fig_w = max(8, len(tsorted) * 0.35)
    fig, ax = plt.subplots(figsize=(fig_w, 4.5), facecolor="white")
    ax.set_facecolor("#EBEBEB")
    ax.grid(axis="y", linestyle="-", color="white", linewidth=0.8, zorder=0)
    ax.grid(axis="x", linestyle="-", color="white", linewidth=0.8, zorder=0)
    ax.margins(x=0.02)
    for method in SCATTER_METHODS:
        means = [sampled_data[method][t][metric].mean()
                 if t in sampled_data.get(method, {}) and len(sampled_data[method][t]) else np.nan
                 for t in tsorted]
        col = COLOUR_MAP.get(method, "#888888")
        ax.plot(x_pos, means, color=col, linewidth=1.4,
                marker=METHOD_MARKERS.get(method, "o"), markersize=6,
                markerfacecolor="white", markeredgecolor=col, markeredgewidth=1.5,
                label=method, zorder=3)
    ax.set_xlim(-0.5, len(tsorted) - 0.5); ax.set_ylim(0.0, y_max)
    ax.set_xticks(x_pos); ax.set_xticklabels(tsorted, rotation=90, ha="center", fontsize=9)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_ylabel(metric, fontsize=12); ax.set_xlabel("Target", fontsize=12)
    title = {"BEDROC": "DUDE-Z Mean BEDROC (α=20) Retrieval Performance",
             "EF1%": "DUDE-Z Mean EF1% Retrieval Performance"}.get(metric, f"DUDE-Z Mean {metric}")
    ax.set_title(title, fontsize=14, pad=6)
    ax.legend(fontsize=10, bbox_to_anchor=(1.01, 1), loc="upper left",
              borderaxespad=0.0, framealpha=0.85)
    plt.tight_layout()
    path = OUT_DIR / f"scatter_{metric.replace('%','pct')}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show()
    log.info("saved %s", path)

scatter("BEDROC")
scatter("EF1%")

## 8 · Wilcoxon signed-rank test — focused on GE-MolSG

In [ ]:
from scipy.stats import wilcoxon

WILCOXON_METHOD = "GE-MolSG"
alpha = 0.05
records = []
for metric in METRICS:
    for baseline in sampled_data:
        if baseline == WILCOXON_METHOD:
            continue
        pf, pb = [], []
        for t in TARGETS:
            if (t in sampled_data[WILCOXON_METHOD] and len(sampled_data[WILCOXON_METHOD].get(t, [])) and
                    t in sampled_data[baseline] and len(sampled_data[baseline].get(t, []))):
                pf.append(sampled_data[WILCOXON_METHOD][t][metric].mean())
                pb.append(sampled_data[baseline][t][metric].mean())
        n = len(pf)
        if n < 4:
            records.append(dict(Metric=metric, Baseline=baseline, N_targets=n,
                                W=np.nan, p_value=np.nan, Significant="—", Direction="insufficient data")); continue
        diffs = np.array(pf) - np.array(pb)
        if np.all(diffs == 0):
            records.append(dict(Metric=metric, Baseline=baseline, N_targets=n,
                                W=np.nan, p_value=np.nan, Significant="—", Direction="identical")); continue
        stat, p = wilcoxon(pf, pb, alternative="two-sided")
        records.append(dict(Metric=metric, Baseline=baseline, N_targets=n,
                            W=round(stat, 3), p_value=round(p, 4),
                            Significant="✓" if p < alpha else "✗",
                            Direction="better" if diffs.mean() > 0 else "worse"))
sig_df = pd.DataFrame(records)
if len(sig_df):
    sig_df = sig_df.sort_values(["Metric", "p_value"])
sig_df.to_csv(OUT_DIR / f"wilcoxon_{WILCOXON_METHOD.replace(' ','_')}.csv", index=False)
print(f"Wilcoxon signed-rank test: {WILCOXON_METHOD} vs all others (α={alpha})\n")
for metric in METRICS:
    if "Metric" not in sig_df.columns:
        break
    sub = sig_df[sig_df.Metric == metric][["Baseline", "N_targets", "W", "p_value", "Significant", "Direction"]]
    print(f"── {metric} {'─'*40}"); print(sub.to_string(index=False)); print()
sig_df